In [2]:
# Run once
!pip install datasets transformers torch python-dotenv

  Obtaining dependency information for python-dotenv from https://files.pythonhosted.org/packages/0b/d7/1959b9648791274998a9c3526f6d0ec8fd2233e4d4acce81bbae76b44b2a/python_dotenv-1.2.2-py3-none-any.whl.metadata



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

# Load token from .env (SAFE)
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

# Login
login(token=hf_token)

In [5]:
from datasets import load_dataset

# Load your dataset (already downloaded)
dataset = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Check data
print("Train rows:", len(dataset["train"]))
print("Languages:", set(dataset["train"]["lang"]))

Train rows: 39178
Languages: {'th', 'it', 'da', 'ko', 'tr', 'en', 'trde', 'sl', 'vi', 'hr', 'sr', 'de', 'id', 'es', 'ja', 'nl', 'iden'}


In [7]:
from collections import defaultdict

# MFR Functions (from official repo)
def counting(data):
    counts = defaultdict(lambda: defaultdict(int))
    for item in data:
        for raw, norm in zip(item["raw"], item["norm"]):
            counts[raw][norm] += 1
    return counts

def mfr(raw_tokens, counts):
    rules = {
        "u": "you",
        "r": "are",
        "ur": "your",
        "bc": "because",
        "bcuz": "because",
        "wut": "what",
        "k": "tidak",
        "gk": "tidak",
        "gak": "tidak"
    }
    result = []
    for t in raw_tokens:
        if t in rules:
            result.append(rules[t])
        elif t in counts:
            result.append(max(counts[t], key=counts[t].get))
        else:
            result.append(t)
    return result

def evaluate(raw_list, gold_list, pred_list):
    tp = fp = fn = 0
    for raw, gold, pred in zip(raw_list, gold_list, pred_list):
        for r, g, p in zip(raw, gold, pred):
            if r == g:
                if p == g:
                    tp += 1
                else:
                    fp += 1
            else:
                if p == g:
                    tp += 1
                else:
                    fn += 1
    total = tp + fp + fn
    err = (fp + fn) / total if total > 0 else 0
    print(f"ERR: {err:.4f} | TP: {tp} | FP: {fp} | FN: {fn}")
    return err

In [8]:
# Train on full data
train = dataset["train"]
val = dataset["validation"]

counts = counting(train)

# Test
print("Smoke Test:")
print(mfr(["because", "u", "r", "funny"], counts))

Smoke Test:
['because', 'you', 'are', 'funny']


In [9]:
import random

def show_random(lang=None):
    data = val
    if lang:
        data = val.filter(lambda x: x["lang"] == lang)
    idx = random.randint(0, len(data)-1)
    noisy = data[idx]["raw"]
    clean = data[idx]["norm"]
    pred = mfr(noisy, counts)
    
    print("="*50)
    print(f"LANG: {data[idx]['lang']}")
    print("NOISY:", noisy)
    print("REAL  :", clean)
    print("PRED  :", pred)
    print("="*50)

# Show 5 random examples
for _ in range(5):
    show_random()

LANG: sl
NOISY: ['sej', 'ponavadi', 'pojem', '3', 'kose', 'in', 'je', 'preveč', '.']
REAL  : ['saj', 'ponavadi', 'pojem', '3', 'kose', 'in', 'je', 'preveč', '.']
PRED  : ['saj', 'ponavadi', 'pojem', '3', 'kose', 'in', 'je', 'preveč', '.']
LANG: id
NOISY: ['Iiihh', 'papanya', 'lucu', 'yah', '@sitiginting_']
REAL  : ['ih', 'papanya', 'lucu', 'ya', '[mention]']
PRED  : ['Iiihh', 'papanya', 'lucu', 'ya', '@sitiginting_']
LANG: sr
NOISY: ['sve', 'dok', 'imam', 'ovakve', 'prijatelje', ',', 'ja', 'sam', 'bogata', ',', 'i', 'nikad', 'sama', '!']
REAL  : ['sve', 'dok', 'imam', 'ovakve', 'prijatelje', ',', 'ja', 'sam', 'bogata', ',', 'i', 'nikad', 'sama', '!']
PRED  : ['sve', 'dok', 'imam', 'ovakve', 'prijatelje', ',', 'ja', 'sam', 'bogata', ',', 'i', 'nikad', 'sama', '!']
LANG: de
NOISY: ['Ich', 'kenne', 'nichts', '(', 'das', 'so', 'schoen', 'ist', 'wie', 'du', ')']
REAL  : ['Ich', 'kenne', 'nichts', '(', 'das', 'so', 'schön', 'ist', 'wie', 'du', ')']
PRED  : ['Ich', 'kenne', 'nichts', '(', 'da